In [1]:
pip install pandas openpyxl bertopic sentence-transformers umap-learn hdbscan scikit-learn

Note: you may need to restart the kernel to use updated packages.


In [2]:
pip install sentence-transformers bertopic scikit-learn

Note: you may need to restart the kernel to use updated packages.


In [3]:
pip install modelscope sentence-transformers bertopic scikit-learn -i https://pypi.tuna.tsinghua.edu.cn/simple

Looking in indexes: https://pypi.tuna.tsinghua.edu.cn/simpleNote: you may need to restart the kernel to use updated packages.



In [4]:
pip install jieba

Note: you may need to restart the kernel to use updated packages.


In [1]:
import os
os.environ['HF_ENDPOINT'] = 'https://hf-mirror.com'  # 国内镜像

from bertopic import BERTopic
from sentence_transformers import SentenceTransformer

In [2]:
import pandas as pd
import re
import random
import jieba
import jieba.posseg as pseg
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.cluster import AgglomerativeClustering
from bertopic import BERTopic
from umap import UMAP
from hdbscan import HDBSCAN
from sklearn.feature_extraction.text import CountVectorizer
from sentence_transformers import SentenceTransformer

In [3]:
# 1 读取新闻数据
# =========================

file_path = r"D:\6601\微博数据抽样1\2025年.xlsx"

df = pd.read_excel(file_path)

print("原始新闻数量:", len(df))


原始新闻数量: 35143


In [4]:
# 1. 配置 jieba（用你的词表）
# =========================

work_terms = [
    "工作","职场","公司","员工","上班","企业","单位",
    "岗位","职业","劳动","白领","加班","同事","打工人","牛马",
    "辞职","离职","跳槽","面试","招聘","简历",
    "996","007","内卷","躺平","摆烂","社畜"
]

mental_terms = [
    "焦虑","抑郁","心理","压力","情绪","心理健康","内耗",
    "精神","心理问题","心理疾病","倦怠","疲惫","累","忙",
    "心理咨询","心理治疗", "失眠","崩溃","破防"
]

# 所有关键词按长度排序（长的先加）
all_terms = sorted(work_terms + mental_terms, key=len, reverse=True)

for word in all_terms:
    jieba.add_word(word, freq=100000)

jieba.initialize()

print(f"已添加 {len(all_terms)} 个自定义词")
print("前10个:", all_terms[:10])

test = "职场心理健康问题导致打工人焦虑"
print(f"\n测试分词: {list(jieba.cut(test))}")

Building prefix dict from the default dictionary ...
Loading model from cache C:\Users\Le'novo\AppData\Local\Temp\jieba.cache
Loading model cost 0.744 seconds.
Prefix dict has been built successfully.


已添加 46 个自定义词
前10个: ['心理健康', '心理问题', '心理疾病', '心理咨询', '心理治疗', '打工人', '996', '007', '工作', '职场']

测试分词: ['职场', '心理健康', '问题', '导致', '打工人', '焦虑']


In [5]:
# 2. 先筛选（用原始文本！）
# =========================

work_pattern = "|".join(work_terms)
mental_pattern = "|".join(mental_terms)

df_filtered = df[
    df["TextContent"].str.contains(work_pattern, na=False) &
    df["TextContent"].str.contains(mental_pattern, na=False)
]

print(f"原始: {len(df)}, 筛选后: {len(df_filtered)}")

原始: 35143, 筛选后: 26254


In [6]:
# 3. 再清洗（只处理筛选后的数据）
# =========================

surnames = "王李张刘陈杨黄赵周吴徐孙马朱胡郭何林罗高郑梁谢宋唐许韩冯邓曹彭曾肖田董袁潘于蒋蔡余杜叶程苏魏吕丁任沈姚卢姜崔钟谭陆汪范金石廖贾夏付方白邹孟熊秦邱江尹薛闫段雷侯龙史黎贺顾毛郝龚邵万钱严覃武戴莫孔向汤关"

def clean_text(text):
    text = str(text)
    text = re.sub(r"http\S+", "", text)
    text = re.sub(r"[^\u4e00-\u9fa5]", " ", text)
    
    # 删除人名
    name_pattern = f"[{surnames}][\u4e00-\u9fa5]{{1,3}}"
    text = re.sub(name_pattern, " ", text)

    # 删除地名：xx市、xx县、xx区、xx镇、xx省
    text = re.sub(r"[\u4e00-\u9fa5]{2,6}(市|县|区|镇|乡|村|省|州|盟)", " ", text)
        
    text = re.sub(r"\s+", " ", text)
    return text.strip()

# 只清洗筛选后的数据
df_filtered["clean_text"] = df_filtered["TextContent"].apply(clean_text)


In [7]:
# 4. 分词（过滤人名）
# =========================

def tokenize(text):
    words = pseg.cut(text)
    # 保留：非人名 + 长度>=1（保留"累""忙""愁"等单字情绪）
    words = [w for w, f in words if f not in ['nr', 'nrt'] and len(w) >= 1]
    return " ".join(words)

docs = df_filtered["clean_text"].tolist()
docs_tokenized = [tokenize(d) for d in docs]


In [8]:
# ==================== 1. 分层停用词表（学术规范版）====================

# 第一层：通用停用词（引用哈工大/百度，约50词）
GENERAL_STOPWORDS = [
    "的", "了", "在", "是", "我", "有", "和", "就", "不", "人", "都",
    "一", "个", "上", "也", "到", "说", "要", "去", "你", "您","会", "着", "而", "而是",
    "这", "那", "之", "与", "及", "等", "如", "同", "约", "可以", "还是", "才能",
    "就是", "现在", "今天", "时候", "因为", "所以", "但是", "然后", "如果", "保证", "真正",
    "不过", "虽然", "一下", "什么", "我们", "此外", "总之", "又", "但", "元", "项", "左右",
    "他", "她", "它", "他们", "她们", "它们", "自己", "大家"
]

# 第二层：语料库特定停用词（基于统计，客观提取）
# 定义必须保留的核心研究词汇（绝不进入停用词表）
CORE_CONCEPTS = [
    # 职场核心词
    "工作","职场","公司","员工","上班","企业","单位",
    "岗位","职业","劳动","白领","加班","同事","打工人","牛马",
    "辞职","离职","跳槽","面试","招聘","简历",
    "996","007","内卷","躺平","摆烂","社畜",
    
    # 心理健康核心词
    "焦虑","抑郁","心理","压力","情绪","心理健康","内耗",
    "精神","心理问题","心理疾病","倦怠","疲惫","累","忙",
    "心理咨询","心理治疗", "失眠","崩溃","破防"
]

def get_corpus_specific_stopwords(docs, threshold=0.3, protected_words=None):
    from collections import Counter
    
    if protected_words is None:
        protected_words = []
    
    all_words = []
    doc_sets = [set(d.split()) for d in docs]
    
    word_doc_count = Counter()
    for doc_set in doc_sets:
        for word in doc_set:
            word_doc_count[word] += 1
    
    n_docs = len(docs)
    
    # 过滤：高频 BUT 不在保护列表中
    corpus_stopwords = [
        w for w, c in word_doc_count.most_common(200)
        if c > n_docs * threshold 
        and len(w) >= 2
        and w not in protected_words  # 关键：排除核心词
    ]
    
    return corpus_stopwords[:20]

# 使用
corpus_specific = get_corpus_specific_stopwords(
    docs_tokenized, 
    threshold=0.3,
    protected_words=CORE_CONCEPTS
)
print(f"语料库特定停用词：{corpus_specific}")

# 第三层：研究特定停用词（基于预分析，透明披露）
STUDY_SPECIFIC_STOPWORDS = [
    # 广告噪音（从预分析发现）
    "鉴定", "亲子", "采样", "胎儿", "基因", "菌群", "过敏原", "亲子鉴定",
    "选择", "这样", "需要", "办理", "检测", "耐受", "正规",
    "咨询", "预约", "服务", "机构", "中心", "地址", "电话", "联系",
    "标准", "费用", "价格", "收费", "免费", "优惠", "活动",
    # 明星娱乐噪音
    "演唱会", "舞台", "唱", "歌手", "歌词", "演员", "品牌", "大使", "代言人",
    # 生活服务噪音
    "火锅", "美食", "零食", "饮食", "餐厅", "食物",
    # 其他噪音（来自微博）
    "正文", "信用", "晚安", "发现", "每日", "善",
    "微博", "展开", "全文", "点击", "查看", "阅读", "转发",
    "评论", "点赞", "关注", "粉丝", "超话", "热搜", "网页链接", "网页", "链接",
]

# 合并完整停用词表
def get_full_stopwords(docs):
    corpus_specific = get_corpus_specific_stopwords(docs, threshold=0.3)
    full_stopwords = list(set(
        GENERAL_STOPWORDS + corpus_specific + STUDY_SPECIFIC_STOPWORDS
    ))
    print(f"\n双重检查核心词：")
    for word in ["工作", "焦虑", "压力", "情绪"]:
        if word in full_stopwords:
            print(f"  ❌ 警告：'{word}'误入选停用词表，已移除")
            full_stopwords.remove(word)
        else:
            print(f"  ✅ '{word}'已保护，不在停用词表中")
            
    print(f"停用词统计：通用{len(GENERAL_STOPWORDS)} + "
          f"语料库特定{len(corpus_specific)} + "
          f"研究特定{len(STUDY_SPECIFIC_STOPWORDS)} = "
          f"总计{len(full_stopwords)}")
    print(f"语料库特定停用词：{corpus_specific[:10]}...")
    return full_stopwords

# ==================== 2. 主分析（完整停用词表）====================

print("=" * 50)
print("主分析：完整停用词表")
print("=" * 50)

full_stopwords = get_full_stopwords(docs_tokenized)

vectorizer_model = CountVectorizer(
    stop_words=full_stopwords,
    ngram_range=(1, 2),
    min_df=3,
    max_df=0.7,
    token_pattern=r"(?u)\b\w+\b"
)

语料库特定停用词：['收起', '自己', '一个']
主分析：完整停用词表

双重检查核心词：
  ❌ 警告：'工作'误入选停用词表，已移除
  ❌ 警告：'焦虑'误入选停用词表，已移除
  ✅ '压力'已保护，不在停用词表中
  ✅ '情绪'已保护，不在停用词表中
停用词统计：通用68 + 语料库特定5 + 研究特定67 = 总计137
语料库特定停用词：['工作', '收起', '焦虑', '自己', '一个']...


In [9]:
# 6 Embedding 模型
# =========================

# 本地加载模型（替换成你下载的文件夹路径）
embedding_model = SentenceTransformer(r"D:\6601\all-MiniLM-L6-v2")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: D:\6601\all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [10]:
# 7 BERTopic 建模
# =========================


umap_model = UMAP(
    n_neighbors=15, 
    n_components=5, 
    metric='cosine', 
    random_state=42,
    min_dist=0.0           # 让聚类更紧密
)

hdbscan_model = HDBSCAN(
    min_cluster_size=200,   # 大幅提高！每个主题至少100条
    min_samples=10,         # 相应提高
    metric='euclidean',
    cluster_selection_method='eom',
    cluster_selection_epsilon=0.2,  # 控制聚类紧密度
    prediction_data=True
)

# 重新训练
topic_model = BERTopic(
    embedding_model=embedding_model,
    vectorizer_model=vectorizer_model,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    #language="multilingual",
    calculate_probabilities=False,
    verbose=True
)

topics, _ = topic_model.fit_transform(docs_tokenized)


2026-03-09 12:27:55,417 - BERTopic - Embedding - Transforming documents to embeddings.


Batches:   0%|          | 0/821 [00:00<?, ?it/s]

2026-03-09 12:38:15,757 - BERTopic - Embedding - Completed ✓
2026-03-09 12:38:15,758 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-03-09 12:38:54,583 - BERTopic - Dimensionality - Completed ✓
2026-03-09 12:38:54,584 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-03-09 12:38:57,350 - BERTopic - Cluster - Completed ✓
2026-03-09 12:38:57,357 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-03-09 12:39:05,450 - BERTopic - Representation - Completed ✓


In [11]:
# 8 查看初始 topics
# =========================

topic_info = topic_model.get_topic_info()

print(topic_info)



   Topic  Count                    Name  \
0     -1    437   -1_住所_所 住所_业务 肠道_微量元素   
1      0  13398           0_价值_成长_发展_事业   
2      1  10720       1_累_工作 焦虑_不想_找 工作   
3      2    372  2_全身_荨麻疹_仅供参考 工作_资格 竭诚   
4      3    337            3_同事_离职_老板_干   
5      4    293           4_创_广州_无 创_样本   
6      5    277            5_歌_首歌_网友_永远   
7      6    210         6_歌_休假_少年_少年 团队   
8      7    210  7_全身_慢性_仅供参考 工作_结果 真实性   

                                      Representation  \
0  [住所, 所 住所, 业务 肠道, 微量元素, 肠道, 肠道 微量元素, 姑娘, 有限公司,...   
1            [价值, 成长, 发展, 事业, 环境, 越, 他人, 核心, 机会, 手机]   
2      [累, 工作 焦虑, 不想, 找 工作, 倦怠, 好好, 干, 好 焦虑, 开心, 同事]   
3  [全身, 荨麻疹, 仅供参考 工作, 资格 竭诚, 此 供, 不能 具备, 很 术, 对 千...   
4          [同事, 离职, 老板, 干, 员工, 累, 重度, 领导, 司机, 重度 焦虑]   
5          [创, 广州, 无 创, 样本, 定, 鉴, 鉴 定, 司法, 个人隐私, 法医]   
6             [歌, 首歌, 网友, 永远, 歌曲, 快乐, 回应, 好好, 写, 演出]   
7     [歌, 休假, 少年, 少年 团队, 团队 长, 热爱, 时代 少年, 朝, 蜉蝣, 兄弟]   
8  [全身, 慢性, 仅供参考 工作, 结果 真实性, 不能 具备, 竭诚 地, 仪器 很, 准...   

        

In [38]:
import pandas as pd
import numpy as np
import random
import time
import os
from tqdm import tqdm
from collections import Counter
import jieba
import gc

# ==================== 配置 ====================

INPUT_FILE = r"D:\6601\微博数据清洗后（13个关键词） (2)\总共\合并结果_2025年12月.xlsx"
OUTPUT_DIR = r"D:\6601\预测结果_全部数据"
BATCH_SIZE = 5000

os.makedirs(OUTPUT_DIR, exist_ok=True)
CHECKPOINT_FILE = f"{OUTPUT_DIR}/processing_checkpoint.txt"

# ==================== 1. 初始化 ====================

print("=" * 70)
print("百万级数据预测 - 保留全部列")
print("=" * 70)

for word in all_terms:
    jieba.add_word(word, freq=100000)
jieba.initialize()

_SURNAMES = set("王李张刘陈杨黄赵周吴徐孙马朱胡郭何林罗高郑梁谢宋唐许韩冯邓曹彭曾肖田董袁潘于蒋蔡余杜叶程苏魏吕丁任沈姚卢姜崔钟谭陆汪范金石廖贾夏付方白邹孟熊秦邱")

def tokenize_single(text):
    if pd.isna(text) or not isinstance(text, str):
        return ""
    words = jieba.lcut(text)
    filtered = [w for w in words if len(w) >= 2 or (len(w) == 1 and w not in _SURNAMES)]
    return " ".join(filtered)

# ==================== 2. 读取Excel ====================

print("\n读取Excel...")
df_full = pd.read_excel(INPUT_FILE)

total_lines = len(df_full)
original_columns = df_full.columns.tolist()  # 保存原始列名
print(f"总行数: {total_lines:,}")
print(f"原始列: {original_columns}")

# ==================== 3. 检查断点续传 ====================

processed_batches = set()
if os.path.exists(CHECKPOINT_FILE):
    with open(CHECKPOINT_FILE, 'r') as f:
        processed_batches = set(int(line.strip()) for line in f if line.strip())
    print(f"已处理: {len(processed_batches)} 批")

# ==================== 4. 分批处理 ====================

n_batches = (total_lines + BATCH_SIZE - 1) // BATCH_SIZE
print(f"总批次数: {n_batches}")

# 新增列名
new_columns = ['clean_text', 'topic_id', 'orientation']
all_columns = original_columns + new_columns

# 初始化结果文件（带全部列）
result_csv = f"{OUTPUT_DIR}/results_with_all_columns.csv"
if not os.path.exists(result_csv):
    pd.DataFrame(columns=all_columns).to_csv(result_csv, index=False, encoding='utf-8-sig')

total_predicted = 0
topic_dist_total = Counter()
start_time = time.time()

for batch_idx in range(n_batches):
    if batch_idx in processed_batches:
        continue
    
    print(f"\n批次 {batch_idx+1}/{n_batches} ({(batch_idx+1)/n_batches*100:.1f}%)")
    
    batch_start = time.time()
    
    # 切片
    start_idx = batch_idx * BATCH_SIZE
    end_idx = min((batch_idx + 1) * BATCH_SIZE, total_lines)
    df_batch = df_full.iloc[start_idx:end_idx].copy()
    
    # 筛选
    text_column = "Text Content"  # 根据实际列名修改
    
    df_filtered = df_batch[
        df_batch[text_column].str.contains(work_pattern, na=False) &
        df_batch[text_column].str.contains(mental_pattern, na=False)
    ]
    
    if len(df_filtered) == 0:
        with open(CHECKPOINT_FILE, 'a') as f:
            f.write(f"{batch_idx}\n")
        continue
    
    # 分词
    print(f"  筛选后{len(df_filtered)}条，分词...")
    texts = df_filtered[text_column].tolist()
    tokenized_list = [tokenize_single(t) for t in tqdm(texts, desc="分词")]
    
    # 预测
    print("  预测...")
    topics_list, _ = topic_model.transform(tokenized_list)
    
    # 添加新列
    df_filtered['clean_text'] = tokenized_list
    df_filtered['topic_id'] = topics_list
    df_filtered['orientation'] = [
        'positive' if t == 0 else 'negative' if t == 1 else 'other'
        for t in topics_list
    ]
    
    # 保存（保留全部原始列 + 新增列）
    df_filtered[all_columns].to_csv(result_csv, mode='a', header=False, index=False, encoding='utf-8-sig')
    
    # 统计
    total_predicted += len(topics_list)
    topic_dist_total.update(topics_list)
    
    # 检查点
    with open(CHECKPOINT_FILE, 'a') as f:
        f.write(f"{batch_idx}\n")
    
    # 清理
    del df_batch, df_filtered
    gc.collect()
    
    # 报告
    batch_time = time.time() - batch_start
    total_time = time.time() - start_time
    print(f"  本批{batch_time/60:.1f}分钟，累计{total_time/3600:.2f}小时")

# ==================== 5. 清理和结果 ====================

del df_full
gc.collect()

print("\n" + "=" * 70)
print("完成！")
print("=" * 70)

print(f"\n总处理: {total_predicted:,}条")
for tid, cnt in sorted(topic_dist_total.items()):
    print(f"  主题{tid}: {cnt:,}条 ({cnt/total_predicted*100:.1f}%)")

positive = topic_dist_total.get(0, 0)
negative = topic_dist_total.get(1, 0)
print(f"\n积极: {positive:,}条")
print(f"消极: {negative:,}条")

print(f"\n✅ 结果保存: {result_csv}")
print(f"   包含列: {all_columns}")

百万级数据预测 - 保留全部列

读取Excel...
总行数: 94,766
原始列: ['Key Word', 'User Name', 'Publish Time', 'Text Content', 'Repost Count', 'Comment Count', 'Like Count', 'Post Link', '来源文件', 'Unnamed: 8', 'Unnamed: 9', 'Unnamed: 10', 4473, 'Unnamed: 11', 4741]
总批次数: 19

批次 1/19 (5.3%)
  筛选后3703条，分词...


分词: 100%|██████████| 3703/3703 [00:06<00:00, 535.87it/s]


  预测...


Batches:   0%|          | 0/116 [00:00<?, ?it/s]

2026-03-09 20:52:57,664 - BERTopic - Dimensionality - Reducing dimensionality of input embeddings.
2026-03-09 20:52:59,983 - BERTopic - Dimensionality - Completed ✓
2026-03-09 20:52:59,983 - BERTopic - Clustering - Approximating new points with `hdbscan_model`
2026-03-09 20:53:00,187 - BERTopic - Cluster - Completed ✓


...

分词: 100%|██████████| 4194/4194 [00:09<00:00, 465.89it/s]


  预测...


Batches:   0%|          | 0/132 [00:00<?, ?it/s]

2026-03-09 21:27:57,764 - BERTopic - Dimensionality - Reducing dimensionality of input embeddings.
2026-03-09 21:28:00,349 - BERTopic - Dimensionality - Completed ✓
2026-03-09 21:28:00,349 - BERTopic - Clustering - Approximating new points with `hdbscan_model`
2026-03-09 21:28:00,612 - BERTopic - Cluster - Completed ✓


  本批2.3分钟，累计0.61小时

完成！

总处理: 75,420条
  主题-1: 480条 (0.6%)
  主题0: 46,597条 (61.8%)
  主题1: 25,865条 (34.3%)
  主题2: 25条 (0.0%)
  主题3: 768条 (1.0%)
  主题4: 455条 (0.6%)
  主题5: 847条 (1.1%)
  主题6: 354条 (0.5%)
  主题7: 29条 (0.0%)

积极: 46,597条
消极: 25,865条

✅ 结果保存: D:\6601\预测结果_全部数据/results_with_all_columns.csv
   包含列: ['Key Word', 'User Name', 'Publish Time', 'Text Content', 'Repost Count', 'Comment Count', 'Like Count', 'Post Link', '来源文件', 'Unnamed: 8', 'Unnamed: 9', 'Unnamed: 10', 4473, 'Unnamed: 11', 4741, 'clean_text', 'topic_id', 'orientation']


In [68]:

# 获取所有有效主题（排除 -1）
topic_info = topic_model.get_topic_info()
all_topics = topic_info[topic_info['Topic'] != -1]['Topic'].tolist()

print(f"当前主题: {all_topics}")

# 获取主题向量（带安全检查）
topic_vectors = []
valid_topic_ids = []

for topic_id in all_topics:
    result = topic_model.get_topic(topic_id)
    
    # 安全检查：确保返回的是列表，不是 False/None
    if not result or not isinstance(result, list):
        print(f"警告：主题{topic_id} 无效，跳过")
        continue
    
    # 提取权重（c-TF-IDF值）
    weights = [weight for _, weight in result[:10]]
    
    # 填充到固定长度
    if len(weights) < 10:
        weights = weights + [0.0] * (10 - len(weights))
    
    topic_vectors.append(weights)
    valid_topic_ids.append(topic_id)

# 转换为numpy数组
topic_vectors = np.array(topic_vectors)
print(f"有效主题数: {len(valid_topic_ids)}")

# 聚类成4组
if len(valid_topic_ids) > 4:
    clustering = AgglomerativeClustering(n_clusters=4, linkage='ward')
    labels = clustering.fit_predict(topic_vectors)
    
    # 按标签分组
    groups = [[], [], [], []]
    for topic_id, label in zip(valid_topic_ids, labels):
        groups[label].append(topic_id)
    
    print(f"自动分组: {groups}")
    
    # 一次性合并
    topic_model.merge_topics(docs_tokenized, topics_to_merge=groups)
else:
    print("主题数已≤4，无需合并")

当前主题: [0, 1, 2, 3, 4, 5, 6, 7, 8]
有效主题数: 9
自动分组: [[0, 1, 3, 5], [7], [2, 6, 8], [4]]


In [69]:
# 最简查看
final_info = topic_model.get_topic_info()
print(final_info)

print("\n各主题核心词：")
for tid in final_info['Topic'][:5]:
    if tid == -1: continue
    words = [w for w, _ in topic_model.get_topic(tid)[:8]]
    cnt = final_info[final_info['Topic']==tid]['Count'].values[0]
    print(f"主题{tid}（{cnt}条）: {words}")

   Topic  Count                      Name  \
0     -1    569  -1_业务 肠道_微量元素_肠道 微量元素_肠道   
1      0  24452             0_成长_教育_目标_过度   
2      1    756       1_全身_症状_仅供参考 工作_荨麻疹   
3      2    286              2_样本_创_鉴_无 创   
4      3    191          3_歌_休假_最 快乐_之前 做   

                                      Representation  \
0  [业务 肠道, 微量元素, 肠道 微量元素, 肠道, 广州 广州, 有限公司, 微博之夜, ...   
1           [成长, 教育, 目标, 过度, 解决, 知识, 处理, 现实, 行动, 减少]   
2  [全身, 症状, 仅供参考 工作, 荨麻疹, 资格 竭诚, 此 供, 参考 排名, 不能 具...   
3         [样本, 创, 鉴, 无 创, 鉴 定, 法医, 司法, 个人隐私, 河北, 孕期]   
4    [歌, 休假, 最 快乐, 之前 做, 不会 让, 就让, 明月, 成立, 可能 让, 青年]   

                                 Representative_Docs  
0  [正规 湿疹 食物   不   耐   受 医院 汇总   附   年 检测 机构 地址 合...  
1  [丫头   只 需 坚持 三个 月   让 看不起 你 的 人 闭嘴   规律 作息 不 熬...  
2  [抚州   家 食物 不 耐受   检测 机构 指引   附   年 最新 检测 攻略   ...  
3  [宜昌   所 本地 孕期 亲子鉴定 医疗机构 梳理   附 全新 鉴定 步骤   宜昌  ...  
4  [话   朝   夜归   真正 意义 上 的 休假 是 在 四年 前   休假 去 了 一...  

各主题核心词：
主题0（24452条）: ['成长', '教育', '目标', '过度', '解决', '知识', '处理

In [ ]:
# 10 写回 Excel
# =========================

df_filtered["topic"] = topic_info
# 自动写入 topic keywords

topic_labels = {
    topic: ", ".join([word for word, _ in topic_model.get_topic(topic)])
    for topic in topic_model.get_topics().keys()
}

df_filtered["topic_keywords"] = df_filtered["topic"].map(topic_labels)

output_path = r"C:\Users\97201\Desktop\6601 分析（2026年3月）\news_topic_results.xlsx"

df_filtered.to_excel(output_path, index=False)

print("结果已保存:", output_path)

结果已保存: C:\Users\97201\Desktop\6601 分析（2026年3月）\news_topic_results.xlsx


In [20]:
# 11 可视化
# =========================

fig = topic_model.visualize_topics()

fig.write_html(
r"D:\6601\预测结果_全部数据\socialmedia_topics_visualization.html"
)

print("BERTopic 完成")

BERTopic 完成
